In [8]:
path = r"..\export\2025-08-04\simulation_1_Kiel.json"



In [11]:
folder_path = r"..\..\export\2025-08-08"

# set path to last created file in folder
import os
import glob
files = glob.glob(os.path.join(folder_path, "*.json"))
if files:
    path = max(files, key=os.path.getctime)
    print(f"Using latest file: {path}")
    

Using latest file: ..\..\export\2025-08-08\simulation_1_Kiel.json


In [12]:

import json
import pandas as pd

with open(path, encoding="utf-8") as f:
    data = json.load(f, )

df = pd.DataFrame(data['operations'])

# Define the desired column order
column_order = [
     'start_date', 'end_date', 'worktype_text', 'machine', 'worktype', 'exa_id', 'field', 'area',
    'distance', 'distanceWorked', 'duration', 'durationWorked', 'fuel',
    'application_type', 'application_category', 'application_name', 'application_amount', 'application_unit',
    'batch', 'model', 'pk'
]
# Normalize the 'fields' column into separate columns and join with the original dataframe (excluding old 'fields')
fields_df = pd.json_normalize(df['fields'])
df_formatted = pd.concat([df.drop(columns=['fields']), fields_df], axis=1)

# Only include columns that exist in df_formatted
ordered_columns = [col for col in column_order if col in df_formatted.columns]
# Add any remaining columns at the end
remaining_columns = [col for col in df_formatted.columns if col not in ordered_columns]
final_columns = ordered_columns + remaining_columns

# Add a total row for numeric columns at the bottom of the table
# Calculate totals BEFORE formatting numbers as strings
exclude_cols = ["application_category", "application_amount", "application_unit", "field", "worktype_text", "area","exa_id", "batch", "model", "pk", "worktype"]
numeric_cols = fields_df.select_dtypes(include=['float', 'int']).columns.intersection(final_columns)

# format start_date and end_date columns with german date format
df_formatted['start_date'] = pd.to_datetime(df_formatted['start_date']).dt.strftime('%d.%m.%Y %H:%M')
df_formatted['end_date'] = pd.to_datetime(df_formatted['end_date']).dt.strftime('%d.%m.%Y %H:%M')


# Exclude specific columns from totals calculation
numeric_cols_total = numeric_cols.difference(exclude_cols)
totals = fields_df[numeric_cols_total].sum(numeric_only=True)
totals_row = {col: totals[col] if col in totals else '' for col in final_columns}
totals_row['worktype_text'] = 'Total'

# Create the DataFrame with the total row
df_with_total = pd.concat(
    [df_formatted[final_columns], pd.DataFrame([totals_row])],
    ignore_index=True
)

df_with_total[numeric_cols] = df_with_total[numeric_cols].apply(
    lambda col: col.map(lambda x: '{:,.0f}'.format(x) if isinstance(x, (int, float)) and pd.notnull(x) else (x if x != '' else ''))
)

df_with_total.style.set_table_styles(
    [
        {'selector': 'th', 'props': [('background-color', '#22223b'), ('color', '#ffffff'), ('font-weight', 'bold'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('padding', '6px'), ('border', '1px solid #ccc'), ('background-color', '#f8f9fa'), ('color', '#22223b')]},
        {'selector': 'tr:nth-child(even) td', 'props': [('background-color', '#e9ecef'), ('color', '#22223b')]},
        {'selector': 'tr:hover td', 'props': [('background-color', '#c9ada7'), ('color', '#22223b')]},
        {'selector': 'tr:last-child td', 'props': [('font-weight', 'bold'), ('background-color', '#b5ead7')]},
    ]
).set_properties(**{'border-color': '#ccc', 'border-width': '1px', 'border-style': 'solid'})


,start_date,end_date,worktype_text,machine,worktype,exa_id,field,area,distance,distanceWorked,duration,durationWorked,fuel,application_type,application_category,application_name,application_amount,application_unit,batch,model,pk
0,17.03.2025 07:37,17.03.2025 20:58,Grubbern,Fendt 719 Vario,6,0,1,15,60,57,"48,060","45,657",193,None,None,None,0,nan,None,pipeline.operation,0
1,26.03.2025 08:05,26.03.2025 22:29,Kreiseln,Fendt 719 Vario,7,0,1,15,50,48,"51,840","49,248",144,None,None,None,0,nan,None,pipeline.operation,0
2,01.04.2025 08:46,02.04.2025 10:52,Separieren,Fendt 719 Vario,28,0,1,15,83,79,"93,960","89,262",399,None,None,None,0,nan,None,pipeline.operation,0
3,10.04.2025 12:14,10.04.2025 15:41,Kali,Fendt 719 Vario,23,0,1,15,8,8,"12,420","11,799",205,None,None,None,0,nan,None,pipeline.operation,0
4,12.04.2025 07:45,12.04.2025 09:15,Pflanzguttransport,Fendt 719 Vario,18,0,1,15,0,0,"5,400","5,130",0,None,None,None,0,nan,None,pipeline.operation,0
5,12.04.2025 12:39,13.04.2025 04:42,Pflanzen,Fendt 719 Vario,26,0,1,15,42,40,"57,780","54,891",167,None,None,None,0,nan,None,pipeline.operation,0
6,16.04.2025 11:53,16.04.2025 19:23,N Düngung,Fendt 719 Vario,23,0,1,15,8,8,"27,000","25,650",15,None,None,None,0,nan,None,pipeline.operation,0
7,25.04.2025 07:50,25.04.2025 13:50,P Düngung,Fendt 719 Vario,23,0,1,15,10,10,"21,600","20,520",15,None,None,None,0,nan,None,pipeline.operation,0
8,02.05.2025 10:03,02.05.2025 13:03,Häufeln,Fendt 719 Vario,29,0,1,15,21,20,"10,800","10,260",43,None,None,None,0,nan,None,pipeline.operation,0
9,03.05.2025 08:46,03.05.2025 11:46,Spritzen,Fendt 719 Vario,14,0,1,15,8,8,"10,800","10,260",16,protection,26,Bandur Artist (2.5+2),68,2,None,pipeline.operation,0
